# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [ ]:
import os

# Async CUDA allocator
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'

# If cuDNN autotune fails, fall back to a safe (but slower) algorithm.
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true' 

### 1.2. Imports

In [ ]:
from _imports import * # Centralized file containing all imports

### 1.3. GPU Management

In [ ]:
# Specify GPU to use (e.g., GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
troo.get_gpu_info()

## 2. Run Parameters 

In [ ]:
NUM_TRIALS = 10
EPOCHS = 1
TOP_K = 1  # Number of top trials to save

mixed_precision.set_global_policy("mixed_float16")

#? Set to an existing path to resume training
RESUME_TRAINING_PATH = None # None or "runs/nas_1" 

In [ ]:
RUN_DIR = RESUME_TRAINING_PATH or troo.create_run_directory(prefix="nas_")
print(f"Run directory: {RUN_DIR}")

## 3. Data Loading and Preprocessing

In [ ]:
def convert_to_sparse_labels(y: np.ndarray) -> np.ndarray:
    """
    Converts beam score targets to sparse integer labels suitable for SparseCategoricalCrossentropy loss.

    Args:
        y (np.ndarray): Original beam scores with shape (N, 8, 32).

    Returns:
        np.ndarray: Array of integer labels with shape (N,), where each label corresponds 
                    to the index (flattened over 8x32) of the maximum score.
    """
    # Reshape the input so that each sample becomes a 1D array (e.g., 256 elements)
    y_flat = y.reshape(y.shape[0], -1)
    # For each sample, return the index of the maximum value
    labels = np.argmax(y_flat, axis=1)
    return labels


In [ ]:
# Define the base directory for data files
DATA_DIR = "./data/s008"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s008_y_train = np.load(beam_output_path)
s008_coord_input = np.load(coord_input_path)
s008_image_input = np.load(image_input_path)
s008_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s008_y_train = s008_y_train.astype(np.float32)
s008_coord_input = s008_coord_input.astype(np.float32)

print(f"Shape before conversion: {s008_y_train.shape}")
s008_y_train = convert_to_sparse_labels(s008_y_train)
print(f"Shape after conversion: {s008_y_train.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {s008_y_train.shape}")
print(f"coord_input shape: {s008_coord_input.shape}")
print(f"image_input shape: {s008_image_input.shape}")
print(f"lidar_input shape: {s008_lidar_input.shape}")

In [ ]:
# Define the base directory for data files
DATA_DIR = "./data/s009"

# Construct full paths to the .npy files
beam_output_path = os.path.join(DATA_DIR, "beam_output", "output_classification.npy")
coord_input_path = os.path.join(DATA_DIR, "coord_input", "coordinates.npy")
image_input_path = os.path.join(DATA_DIR, "image_input", "inputs.npy")
lidar_input_path = os.path.join(DATA_DIR, "lidar_input", "input.npy")

# Load the data from the .npy files
s009_y = np.load(beam_output_path)
s009_coord_input = np.load(coord_input_path)
s009_image_input = np.load(image_input_path)
s009_lidar_input = np.load(lidar_input_path)

# Cast target beam outputs to float - REMOVING USELESS IMAG PART
s009_y = s009_y.astype(np.float32)
s009_coord_input = s009_coord_input.astype(np.float32)

print(f"Shape before conversion: {s009_y.shape}")
s009_y = convert_to_sparse_labels(s009_y)
print(f"Shape after conversion: {s009_y.shape}")

# Print the shapes of the loaded data
print(f"y_train shape: {s009_y.shape}")
print(f"coord_input shape: {s009_coord_input.shape}")
print(f"image_input shape: {s009_image_input.shape}")
print(f"lidar_input shape: {s009_lidar_input.shape}")

In [ ]:
# # ----------------------- Subsample dataset for testing ---------------------- #
# SAMPLE_SIZE = 50  # Use a subset of x samples for testing
# s008_y_train = s008_y_train[:SAMPLE_SIZE]
# s008_coord_input = s008_coord_input[:SAMPLE_SIZE]
# s008_image_input = s008_image_input[:SAMPLE_SIZE]
# s008_lidar_input = s008_lidar_input[:SAMPLE_SIZE]

# # Print the shapes of the loaded data
# print(f"y_train s008 shape: {s008_y_train.shape}")
# print(f"coord_input s008 shape: {s008_coord_input.shape}")
# print(f"image_input s008 shape: {s008_image_input.shape}")
# print(f"lidar_input s008 shape: {s008_lidar_input.shape}")

# s009_y_train = s009_y_train[:SAMPLE_SIZE]
# s009_coord_input = s009_coord_input[:SAMPLE_SIZE]
# s009_image_input = s009_image_input[:SAMPLE_SIZE]
# s009_lidar_input = s009_lidar_input[:SAMPLE_SIZE]

# print(f"y_train s009 shape: {s009_y_train.shape}")
# print(f"coord_input s009 shape: {s009_coord_input.shape}")
# print(f"image_input s009 shape: {s009_image_input.shape}")
# print(f"lidar_input s009 shape: {s009_lidar_input.shape}")

## 4. Getters

### 4.1. Regularizers

In [ ]:
def get_regularizer(trial: optuna.Trial, name: str) -> Optional[tf.keras.regularizers.Regularizer]:
    """
    Suggests a regularization strategy using Optuna and returns the corresponding Keras regularizer.
    
    Args:
        trial (optuna.Trial): Optuna trial object used to sample the regularizer.
        name (str): Unique identifier for this regularizer parameter (used as key).

    Returns:
        Optional[tf.keras.regularizers.Regularizer]: The selected Keras regularizer instance,
        or `None` if "none" was selected.
    """
    # Suggest a regularizer type
    reg_type: str = trial.suggest_categorical(
        name,
        [
            "none",
            "l1",
            "l2",
            "l1l2",
            # "orthogonal",  #! only works for rank-2 tensors
        ],
    )

    # Map each regularizer name to a corresponding Keras regularizer instance
    regularizer_map: Dict[str, Optional[tf.keras.regularizers.Regularizer]] = {
        "none": None,
        "l1": regularizers.L1(l1=0.01),
        "l2": regularizers.L2(l2=0.01),
        "l1l2": regularizers.L1L2(l1=0.01, l2=0.01),
        "orthogonal": regularizers.OrthogonalRegularizer(factor=0.01, mode="rows"),
    }

    # Return the appropriate regularizer, or None if not found
    return regularizer_map.get(reg_type, None)

### 4.2. Activation Functions

In [ ]:
def get_activation(trial: Any, name: str) -> Union[str, Callable[..., layers.Layer]]:
    """
    Suggests an activation function from a predefined list using Optuna.

    Args:
        trial (Any): The Optuna trial instance used to suggest a value.
        name (str): A unique name for this hyperparameter (e.g., "layer_1_activation").

    Returns:
        Union[str, Callable[..., layers.Layer]]: A string representing the activation function.
        This can be passed directly into a Keras layer's `activation=` argument.
    """
    return trial.suggest_categorical(
        name,
        [
            "relu",
            "tanh",
            "sigmoid",  # Logistic
            "elu", 
            "swish",  # x * sigmoid(x)
            "leaky_relu",
        ],
    )

### 4.3. Optimizers

In [ ]:
def get_optimizer(trial: optuna.Trial) -> tf.keras.optimizers.Optimizer:
    """
    Suggests and returns a TensorFlow optimizer with a trial-based learning rate.

    Args:
        trial (optuna.Trial): Optuna trial object used for hyperparameter suggestion.

    Returns:
        tf.keras.optimizers.Optimizer: An instance of the selected optimizer.
    """
    # Suggest optimizer name from a predefined categorical set
    optimizer_name = trial.suggest_categorical(
        "optimizer",
        [
            "AdamW",
            "SGD",
            "Adam",
            "RMSprop",
            "Nadam",
            "Lion",
        ],
    )

    # Suggest learning rate on a logarithmic scale between 1e-5 and 1e-2
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Mapping of optimizer names to their TensorFlow classes
    optimizer_map: Dict[str, Type[tf.keras.optimizers.Optimizer]] = {
        "Adam": optimizers.Adam,
        "AdamW": optimizers.AdamW,
        "SGD": optimizers.SGD,
        "RMSprop": optimizers.RMSprop,
        "Nadam": optimizers.Nadam,
        "Lion": optimizers.Lion,
    }

    # Raise error if selected optimizer is not supported in the current context
    if optimizer_name not in optimizer_map:
        raise ValueError(
            f"Optimizer '{optimizer_name}' is not supported. "
            f"Supported optimizers are: {list(optimizer_map.keys())}."
        )

    # Instantiate and return the selected optimizer with suggested learning rate
    return optimizer_map[optimizer_name](learning_rate=learning_rate)

### 4.4. Callbacks

In [ ]:
def get_callbacks(trial: optuna.Trial, checkpoint_dir: str) -> List[tf.keras.callbacks.Callback]:
    """
    Constructs and returns a list of Keras callbacks tailored for Optuna trials.

    Args:
        trial (optuna.Trial): The current Optuna trial object.
        checkpoint_dir (str): Directory where model weights will be saved.

    Returns:
        List[tf.keras.callbacks.Callback]: A list of callbacks to pass into `model.fit()`.
    """
    # Construct path for saving weights for this specific trial
    checkpoint_path: str = os.path.join(checkpoint_dir, f"trial_{trial.number}.weights.h5")

    # Metric to monitor for early stopping and checkpointing
    monitor: str = "val_loss"

    # Stop training early if no improvement in validation loss for N epochs
    early_stopping = callbacks.EarlyStopping(
        monitor=monitor,
        patience=6,  # number of epochs to wait
        restore_best_weights=True,
        verbose=1,
    )

    # Reduce learning rate if validation loss plateaus
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor=monitor,
        patience=3,  # how many epochs to wait before reducing LR
        factor=0.2,  # reduce LR by this factor
        min_lr=1e-6,  # don't reduce below this
        verbose=1,
    )

    # Save only the best model weights based on monitored metric
    model_checkpoint = callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor=monitor,
        save_best_only=True,  # only save weights if val_loss improves
        save_weights_only=True,  # save only the weights (not full model)
        verbose=0,
    )

    #! ——————— WARNING: the callbacks below do not work with multi-objective —————— !#
    # Custom callback to prune trial if NaN loss is encountered
    nan_pruner_callback = NanLossPrunerCallback(trial)

    # Optuna's built-in pruning callback for early trial termination
    pruning_callback = KerasPruningCallback(trial, monitor)
    #! ———————————————————————————————————————————————————————————————————————————— !#

    # Return the complete list of callbacks
    return [early_stopping, reduce_lr, model_checkpoint, nan_pruner_callback, pruning_callback]

### 4.5. Scalers

In [ ]:
def get_scaler(
    trial: optuna.Trial,
) -> Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
    """
    Suggests and returns a scikit-learn scaler based on Optuna hyperparameter selection.

    Args:
        trial (optuna.Trial): Optuna trial object used to suggest hyperparameters.

    Returns:
        Union[StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer, PowerTransformer]:
            Instantiated scaler object from scikit-learn.
    """
    # Suggest a scaler name from the list of supported options
    scaler_name = trial.suggest_categorical(
        "scaler",
        [
            "StandardScaler",  # For normally-distributed data
            "MinMaxScaler_-1_1",  # Normalize to [-1, 1] range
            "MinMaxScaler_0_1",  # Normalize to [0, 1] range
            "RobustScaler",  # For data with outliers
            "QuantileTransformer",  # For non-normal or skewed data
            "PowerTransformer",  # For heavy-tailed or skewed data
        ],
    )

    # Return the appropriate scaler instance based on selection
    if scaler_name == "StandardScaler":
        return StandardScaler()
    elif scaler_name == "RobustScaler":
        return RobustScaler()
    elif scaler_name == "QuantileTransformer":
        return QuantileTransformer(output_distribution="normal")
    elif scaler_name == "PowerTransformer":
        return PowerTransformer(method="yeo-johnson")
    elif scaler_name == "MinMaxScaler_0_1":
        return MinMaxScaler(feature_range=(0, 1))
    elif scaler_name == "MinMaxScaler_-1_1":
        return MinMaxScaler(feature_range=(-1, 1))

    # Catch invalid or unknown choices
    else:
        raise ValueError(f"Unknown scaler selected: {scaler_name}")

## 5. Layers Builders

### 5.1. DNN

In [ ]:
def build_funnel_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    max_decay_factor: float = 0.9,
    min_decay_factor: float = 0.1,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn",
) -> layers.Layer:
    """
    Builds a fully connected DNN funnel architecture, gradually reducing unit counts across layers.

    Includes optional batch normalization, dropout, and two styles of residual connections.

    Logic:
        -> Suggest number of layers and initial unit count
        -> Suggest exponential decay factor
        -> For each layer:
            -> Create Dense layer with decayed unit count
            -> Apply optional BatchNormalization and Dropout
            -> Handle residual connections ("beside" or "all")
        -> Return the final output tensor

    Args:
        trial (Any): The Optuna trial instance used to suggest hyperparameters.
        x (layers.Layer): Input tensor from previous layers.
        max_layers (int): Maximum number of Dense layers.
        max_units (int): Maximum number of units in the first Dense layer.
        min_units (int): Minimum number of units for the first layer.
        step (int): Unit count step size for hyperparameter sampling.
        max_decay_factor (float): Max exponential decay rate for unit count.
        min_decay_factor (float): Min exponential decay rate for unit count.
        try_batch_norm (bool): Whether to allow trial-based BatchNormalization layers.
        use_regularization (bool): Whether to apply kernel/bias/activity regularization.
        residual_method (Optional[str]): Residual connection method:
            - "beside": connect previous output to current
            - "all": connect all previous outputs to current
        custom_name (str): Prefix used for layer naming.

    Returns:
        layers.Layer: Output tensor after all funnel blocks.

    Example:
        output = build_funnel_dnn(trial, input_tensor, try_batch_norm=True, residual_method="all")
    """
    # Suggest number of DNN layers and unit count for the first layer
    dnn_layers = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units_layer_0 = trial.suggest_int(f"{custom_name}_units_layer_0", min_units, max_units, step=step)

    # Suggest decay factor for reducing units across layers
    decay_factor = trial.suggest_float(
        f"{custom_name}_decay_factor", min_decay_factor, max_decay_factor, step=0.1
    )

    residual_dense = None  # For 'beside' residual connections
    skip_connections_dense = []  # For 'all' residual connections

    # Construct each DNN layer
    for i in range(dnn_layers):
        # Decay the number of units per layer using exponential decay
        units = units_layer_0 if i == 0 else max(16, int(units_layer_0 * (decay_factor**i)))

        # Suggest activation function
        activation = get_activation(trial, f"{custom_name}_activation_layer_{i}")

        # Apply optional regularizers
        dnn_kernel_regularizer = (
            get_regularizer(trial, f"dnn_kernel_regularizer_layer_{i}") if use_regularization else None
        )
        dnn_bias_regularizer = (
            get_regularizer(trial, f"dnn_bias_regularizer_layer_{i}") if use_regularization else None
        )
        dnn_activity_regularizer = (
            get_regularizer(trial, f"dnn_activity_regularizer_layer_{i}") if use_regularization else None
        )

        # Create dense layer
        x = layers.Dense(
            units=units,
            activation=activation,
            name=f"{custom_name}_dense_{i}",
            kernel_regularizer=dnn_kernel_regularizer,
            bias_regularizer=dnn_bias_regularizer,
            activity_regularizer=dnn_activity_regularizer,
        )(x)

        # Optionally apply BatchNormalization
        if try_batch_norm and trial.suggest_categorical(
            f"{custom_name}_use_batch_norm_layer_{i}", [True, False]
        ):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{i}")(x)

        # Apply dropout with trial-suggested rate
        dropout_rate = trial.suggest_float(f"{custom_name}_dropout_layer_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(dropout_rate, name=f"{custom_name}_dropout_{i}")(x)

        # ————————————— Residual Connection Handling ————————————— #
        if residual_method == "beside":
            if i == 0:
                residual_dense = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}", [True, False]):
                    # Adjust shape if necessary before adding
                    if residual_dense.shape[-1] != x.shape[-1]:
                        residual_dense = layers.Dense(
                            units=x.shape[-1], activation=None, name=f"{custom_name}_residual_dense_{i}"
                        )(residual_dense)
                    x = layers.Add(name=f"{custom_name}_residual_add_{i}")([x, residual_dense])
                    residual_dense = x
                else:
                    residual_dense = x

        elif residual_method == "all":
            if i == 0:
                skip_connections_dense = [x]
            else:
                residuals_to_add = []
                for j, prev in enumerate(skip_connections_dense):
                    if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}_{j}", [True, False]):
                        adjusted_prev = prev

                        # Project to match shape if needed
                        if adjusted_prev.shape[-1] != x.shape[-1]:
                            adjusted_prev = layers.Dense(
                                units=x.shape[-1],
                                activation=None,
                                name=f"{custom_name}_skip_residual_dense_{i}_{j}",
                            )(adjusted_prev)
                        residuals_to_add.append(adjusted_prev)

                # If residuals exist, add them
                if residuals_to_add:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals_to_add)
                skip_connections_dense.append(x)

    # Return the final tensor after all funnel layers
    return x


def build_constant_width_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn_const",
) -> layers.Layer:
    """
    Builds a fully-connected DNN with constant width across all layers.

    Logic:
        -> Sample total layer count and fixed unit width
        -> For each layer:
            -> Apply Dense -> (optional) BatchNorm -> Dropout
            -> Optionally apply residual connections ("beside" or "all")

    Args:
        trial (Any): Optuna trial for hyperparameter sampling.
        x (layers.Layer): Input tensor.
        max_layers (int): Maximum number of layers.
        max_units (int): Max units per layer.
        min_units (int): Min units per layer.
        step (int): Sampling step for unit width.
        try_batch_norm (bool): Whether to allow BatchNorm layers.
        use_regularization (bool): Whether to apply regularizers.
        residual_method (Optional[str]): One of {"beside", "all", None}.
        custom_name (str): Prefix for layer/hyperparameter naming.

    Returns:
        layers.Layer: Output tensor after building the model.
    """
    layers_count = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units = trial.suggest_int(f"{custom_name}_units", min_units, max_units, step=step)

    residual_buffer = None
    skip_buffers = []

    for i in range(layers_count):
        activation = get_activation(trial, f"{custom_name}_activation_{i}")
        kernel_reg = get_regularizer(trial, f"{custom_name}_kernel_reg_{i}") if use_regularization else None
        bias_reg = get_regularizer(trial, f"{custom_name}_bias_reg_{i}") if use_regularization else None
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_reg_{i}") if use_regularization else None
        )

        x = layers.Dense(
            units=units,
            activation=activation,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_dense_{i}",
        )(x)

        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_bn_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_bn_{i}")(x)

        rate = trial.suggest_float(f"{custom_name}_dropout_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(rate, name=f"{custom_name}_dropout_{i}")(x)

        # Residual logic
        if residual_method == "beside":
            if i == 0:
                residual_buffer = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_res_{i}", [True, False]):
                    if residual_buffer.shape[-1] != x.shape[-1]:
                        residual_buffer = layers.Dense(
                            x.shape[-1], activation=None, name=f"{custom_name}_res_dense_{i}"
                        )(residual_buffer)
                    x = layers.Add(name=f"{custom_name}_res_add_{i}")([x, residual_buffer])
                residual_buffer = x

        elif residual_method == "all":
            if i == 0:
                skip_buffers = [x]
            else:
                residuals = []
                for j, prev in enumerate(skip_buffers):
                    if trial.suggest_categorical(f"{custom_name}_use_res_{i}_{j}", [True, False]):
                        if prev.shape[-1] != x.shape[-1]:
                            prev = layers.Dense(
                                x.shape[-1], activation=None, name=f"{custom_name}_skip_proj_{i}_{j}"
                            )(prev)
                        residuals.append(prev)
                if residuals:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals)
                skip_buffers.append(x)

    return x


def build_inverted_funnel_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn_inv_funnel",
) -> layers.Layer:
    """
    Builds a bottleneck DNN: narrow → wide → narrow.

    Logic:
        -> Sample edge and center layer widths
        -> Interpolate each layer width based on distance from midpoint
        -> Apply Dense + (optional) BatchNorm + Dropout
        -> Optionally apply residuals ("beside" or "all")

    Args:
        trial (Any): Optuna trial instance.
        x (layers.Layer): Input tensor.
        max_layers (int): Max total layers.
        max_units (int): Max width at center.
        min_units (int): Min width at edges.
        step (int): Unit sampling step.
        try_batch_norm (bool): Whether to sample BatchNorm.
        use_regularization (bool): Whether to apply regularizers.
        residual_method (Optional[str]): See other functions.
        custom_name (str): Naming prefix.

    Returns:
        layers.Layer: Final tensor.
    """
    layers_count = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units_edge = trial.suggest_int(f"{custom_name}_units_edge", min_units, max_units, step=step)
    units_mid = trial.suggest_int(f"{custom_name}_units_mid", units_edge, max_units, step=step)
    mid_index = (layers_count - 1) / 2.0

    residual_buffer = None
    skip_buffers = []

    for i in range(layers_count):
        distance = abs(i - mid_index) / mid_index if mid_index != 0 else 0.0
        units = int(units_mid - (units_mid - units_edge) * distance)
        units = max(16, units)

        activation = get_activation(trial, f"{custom_name}_activation_{i}")
        kernel_reg = get_regularizer(trial, f"{custom_name}_kernel_reg_{i}") if use_regularization else None
        bias_reg = get_regularizer(trial, f"{custom_name}_bias_reg_{i}") if use_regularization else None
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_reg_{i}") if use_regularization else None
        )

        x = layers.Dense(
            units=units,
            activation=activation,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_dense_{i}",
        )(x)

        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_bn_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_bn_{i}")(x)

        rate = trial.suggest_float(f"{custom_name}_dropout_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(rate, name=f"{custom_name}_dropout_{i}")(x)

        # Residual connection logic reused from constant-width
        if residual_method == "beside":
            if i == 0:
                residual_buffer = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_res_{i}", [True, False]):
                    if residual_buffer.shape[-1] != x.shape[-1]:
                        residual_buffer = layers.Dense(
                            x.shape[-1], activation=None, name=f"{custom_name}_res_dense_{i}"
                        )(residual_buffer)
                    x = layers.Add(name=f"{custom_name}_res_add_{i}")([x, residual_buffer])
                residual_buffer = x

        elif residual_method == "all":
            if i == 0:
                skip_buffers = [x]
            else:
                residuals = []
                for j, prev in enumerate(skip_buffers):
                    if trial.suggest_categorical(f"{custom_name}_use_res_{i}_{j}", [True, False]):
                        if prev.shape[-1] != x.shape[-1]:
                            prev = layers.Dense(
                                x.shape[-1], activation=None, name=f"{custom_name}_skip_proj_{i}_{j}"
                            )(prev)
                        residuals.append(prev)
                if residuals:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals)
                skip_buffers.append(x)

    return x


def build_hourglass_dnn(
    trial: Any,
    x: layers.Layer,
    max_layers: int = 10,
    max_units: int = 2048,
    min_units: int = 128,
    step: int = 128,
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "dnn_hourglass",
) -> layers.Layer:
    """
    Build an hourglass-shaped fully-connected DNN.

    Logic:
        -> Sample total layers, edge/mid units, and plateau width
        -> Split into: ramp-up → plateau → ramp-down
        -> For each layer: interpolate unit size and apply core logic

    Args:
        trial (Any): Optuna trial.
        x (layers.Layer): Input.
        max_layers (int): Max layers.
        max_units (int): Max (plateau) units.
        min_units (int): Min (edge) units.
        step (int): Step size.
        try_batch_norm (bool): Whether to enable batch norm.
        use_regularization (bool): Apply regularizers if True.
        residual_method (Optional[str]): Residual strategy.
        custom_name (str): Prefix for naming.

    Returns:
        layers.Layer: Final tensor after processing.
    """
    layers_count = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)
    units_edge = trial.suggest_int(f"{custom_name}_units_edge", min_units, max_units, step=step)
    units_mid = trial.suggest_int(f"{custom_name}_units_mid", units_edge, max_units, step=step)
    plateau = trial.suggest_int(f"{custom_name}_plateau", 1, layers_count)

    pre = (layers_count - plateau) // 2
    post = layers_count - plateau - pre

    residual_buffer = None
    skip_buffers = []

    for i in range(layers_count):
        if i < pre:
            factor = i / pre if pre > 0 else 1.0
            units = int(units_edge + (units_mid - units_edge) * factor)
        elif i < pre + plateau:
            units = units_mid
        else:
            down_idx = i - (pre + plateau)
            factor = down_idx / post if post > 0 else 1.0
            units = int(units_mid - (units_mid - units_edge) * factor)

        units = max(16, units)

        activation = get_activation(trial, f"{custom_name}_activation_{i}")
        kernel_reg = get_regularizer(trial, f"{custom_name}_kernel_reg_{i}") if use_regularization else None
        bias_reg = get_regularizer(trial, f"{custom_name}_bias_reg_{i}") if use_regularization else None
        activity_reg = (
            get_regularizer(trial, f"{custom_name}_activity_reg_{i}") if use_regularization else None
        )

        x = layers.Dense(
            units=units,
            activation=activation,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            activity_regularizer=activity_reg,
            name=f"{custom_name}_dense_{i}",
        )(x)

        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_bn_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_bn_{i}")(x)

        rate = trial.suggest_float(f"{custom_name}_dropout_{i}", 0.0, 0.5, step=0.1)
        x = layers.Dropout(rate, name=f"{custom_name}_dropout_{i}")(x)

        # Residual logic same as before
        if residual_method == "beside":
            if i == 0:
                residual_buffer = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_res_{i}", [True, False]):
                    if residual_buffer.shape[-1] != x.shape[-1]:
                        residual_buffer = layers.Dense(
                            x.shape[-1], activation=None, name=f"{custom_name}_res_dense_{i}"
                        )(residual_buffer)
                    x = layers.Add(name=f"{custom_name}_res_add_{i}")([x, residual_buffer])
                residual_buffer = x

        elif residual_method == "all":
            if i == 0:
                skip_buffers = [x]
            else:
                residuals = []
                for j, prev in enumerate(skip_buffers):
                    if trial.suggest_categorical(f"{custom_name}_use_res_{i}_{j}", [True, False]):
                        if prev.shape[-1] != x.shape[-1]:
                            prev = layers.Dense(
                                x.shape[-1], activation=None, name=f"{custom_name}_skip_proj_{i}_{j}"
                            )(prev)
                        residuals.append(prev)
                if residuals:
                    x = layers.Add(name=f"{custom_name}_add_{i}")([x] + residuals)
                skip_buffers.append(x)

    return x

### 5.2. CNN

In [ ]:
def build_funnel_cnn2d(
    trial: optuna.Trial,
    x: layers.Layer,
    max_layers: int = 5,
    max_kernel: int = 10,
    max_filters: int = 256,
    min_filters: int = 32,
    filter_step: int = 32,
    min_filter_decay_factor: float = 0.1,
    max_filter_decay_factor: float = 0.9,
    min_kernel_decay_factor: float = 0.1,
    max_kernel_decay_factor: float = 0.9,
    pool_size: Tuple[int, int] = (2, 2),
    try_batch_norm: bool = False,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    custom_name: str = "cnn",
) -> layers.Layer:
    """
    Builds a 2D convolutional neural network (CNN) funnel with max pooling and optional residual connections.

    Logic:
        -> Suggest number of layers and initial filter/kernel parameters
        -> For each layer:
            -> Apply Conv2D with decayed filter/kernel sizes
            -> Apply optional batch normalization
            -> Handle residual connections if enabled
            -> Apply MaxPooling2D

    Args:
        trial (optuna.Trial): Optuna trial object for hyperparameter suggestion.
        x (layers.Layer): Input Keras layer/tensor.
        max_layers (int): Maximum number of convolutional blocks.
        max_kernel (int): Maximum kernel size for the first layer.
        max_filters (int): Maximum number of filters for the first layer.
        min_filters (int): Minimum number of filters.
        filter_step (int): Step size for filter number suggestion.
        min_filter_decay_factor (float): Lower bound on exponential decay of filters.
        max_filter_decay_factor (float): Upper bound on exponential decay of filters.
        min_kernel_decay_factor (float): Lower bound on exponential decay of kernel size.
        max_kernel_decay_factor (float): Upper bound on exponential decay of kernel size.
        pool_size (Tuple[int, int]): Pooling size for MaxPooling2D.
        try_batch_norm (bool): Whether to optionally apply batch normalization.
        use_regularization (bool): Whether to use kernel/bias/activity regularization.
        residual_method (Optional[str]): Residual method to use ("beside", "all", or None).
        custom_name (str): Prefix used to name layers for traceability.

    Returns:
        layers.Layer: Output tensor after all CNN and pooling blocks.

    Example:
        output = build_cnn2d(trial, input_tensor, try_batch_norm=True, residual_method="beside")
    """
    # Sample number of convolutional blocks
    cnn_layers = trial.suggest_int(f"{custom_name}_layers", 1, max_layers)

    # Sample initial filter count for first block
    cnn_filters_layer_0 = trial.suggest_int(
        f"{custom_name}_filters_layer_0", min_filters, max_filters, step=filter_step
    )

    # Sample decay rates for filters and kernels across layers
    filters_decay_factor = trial.suggest_float(
        f"{custom_name}_filter_decay_factor", min_filter_decay_factor, max_filter_decay_factor, step=0.1
    )
    kernel_decay_factor = trial.suggest_float(
        f"{custom_name}_kernel_decay_factor", min_kernel_decay_factor, max_kernel_decay_factor, step=0.1
    )

    residual_cnn = None  # Used for 'beside' residual connections
    skip_connections_cnn = []  # Used for 'all' residual connections

    # Build each CNN block
    for i in range(cnn_layers):
        # Calculate decayed number of filters
        filters = (
            cnn_filters_layer_0 if i == 0 else max(16, int(cnn_filters_layer_0 * (filters_decay_factor**i)))
        )

        # Calculate decayed kernel size (upper bound)
        kernel_limit = int(max_kernel * (kernel_decay_factor**i))

        # Suggest kernel dimensions within the limited range
        kernel_size = (
            trial.suggest_int(f"{custom_name}_kernel_height_{i}", 1, max(1, kernel_limit)),
            trial.suggest_int(f"{custom_name}_kernel_width_{i}", 1, max(1, kernel_limit)),
        )

        # Choose activation function for this block
        activation = get_activation(trial, f"{custom_name}_activation_layer_{i}")

        # Get regularizers if requested
        cnn_kernel_regularizer = (
            get_regularizer(trial, f"cnn_kernel_regularizer_layer_{i}") if use_regularization else None
        )
        cnn_bias_regularizer = (
            get_regularizer(trial, f"cnn_bias_regularizer_layer_{i}") if use_regularization else None
        )
        cnn_activity_regularizer = (
            get_regularizer(trial, f"cnn_activity_regularizer_layer_{i}") if use_regularization else None
        )

        # Apply convolutional layer
        x = layers.Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            activation=activation,
            padding="same",
            name=f"{custom_name}_conv2d_{i}",
            kernel_regularizer=cnn_kernel_regularizer,
            bias_regularizer=cnn_bias_regularizer,
            activity_regularizer=cnn_activity_regularizer,
        )(x)

        # Optionally apply BatchNormalization
        if try_batch_norm and trial.suggest_categorical(f"{custom_name}_use_batch_norm_layer_{i}", [True, False]):
            x = layers.BatchNormalization(name=f"{custom_name}_batch_norm_{i}")(x)

        # ——————— Handle Residual Connections ——————— #
        if residual_method == "beside":
            if i == 0:
                residual_cnn = x
            else:
                if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}", [True, False]):
                    target_channels = x.shape[-1]
                    # Align channels if needed
                    if residual_cnn.shape[-1] != target_channels:
                        residual_cnn = layers.Conv2D(
                            filters=target_channels,
                            kernel_size=(1, 1),
                            padding="same",
                            activation=None,
                            name=f"{custom_name}_residual_conv2d_{i}",
                        )(residual_cnn)
                    # Add residual
                    x = layers.Add()([x, residual_cnn])
                    residual_cnn = x
                else:
                    residual_cnn = x

        elif residual_method == "all":
            if i == 0:
                skip_connections_cnn = [x]
            else:
                residuals_to_add = []
                for j, prev in enumerate(skip_connections_cnn):
                    if trial.suggest_categorical(f"{custom_name}_use_residual_layer_{i}_{j}", [True, False]):
                        target_channels = x.shape[-1]
                        adjusted_prev = prev
                        # Align channel dimensions if needed
                        if adjusted_prev.shape[-1] != target_channels:
                            adjusted_prev = layers.Conv2D(
                                filters=target_channels,
                                kernel_size=(1, 1),
                                padding="same",
                                activation=None,
                                name=f"{custom_name}_skip_residual_conv2d_{i}_{j}",
                            )(adjusted_prev)
                        residuals_to_add.append(adjusted_prev)

                # Add residuals to current output
                if residuals_to_add:
                    x = layers.Add()([x] + residuals_to_add)
                skip_connections_cnn.append(x)

        # Apply MaxPooling2D to reduce spatial resolution
        x = layers.MaxPooling2D(
            pool_size=pool_size,
            name=f"{custom_name}_maxpool_{i}"
        )(x)

    # Return the final output tensor after all blocks
    return x

## 6. Objective Function

In [ ]:
def objective(
    trial: optuna.Trial,
    X: List[np.ndarray],
    y: List[np.ndarray],
    checkpoint_dir: str,
    model_dir: str,
    fig_dir: str,
    logs_dir: str,
    epochs: int = 50,
    size_penalizer: Optional[str] = None,
    use_regularization: bool = False,
    residual_method: Optional[str] = None,
    show_summary: bool = False,
    plot_model: bool = False,
) -> float:
    """
    Objective function for Optuna to optimize a Neural Network NN on any-input data.

    Args:
        trial (optuna.Trial): Current trial for hyperparameter suggestions.
        X (List[np.ndarray]): List of input arrays.
        y (List[np.ndarray]): List of label arrays.
        checkpoint_dir (str): Path to store checkpoint files.
        model_dir (str): Path to store full models.
        fig_dir (str): Path to store plots.
        logs_dir (str): Path to store logs.
        epochs (int): Number of training epochs.
        size_penalizer (Optional[str]): type of penalizer to use:
            - "params": Penalizes based on the number of parameters.
            - "flops": Penalizes based on the number of FLOPs.
            - None: No penalization is applied.
        use_regularization (bool): If True, adds regularization (e.g., L1/L2) to layers to prevent overfitting.
        residual_method (Optional[str]): tyoe of residual connection to use:
            - "beside": Adds residual connections between consecutive layers.
            - "all": test residual connections between all layers.
            - None: No residual connections are applied.
        show_summary (bool): If True, display the model summary.
        plot_model (bool): If True, display a plot of the model architecture.

    Returns:
        float: Final validation loss (optionally penalized) used for optimization.
    """

    # Each trial gets a different seed to split the data
    np.random.seed(trial.number)
    tf.random.set_seed(trial.number)

    # ————————————————————————————— Prepare the Data ————————————————————————————— #
    s008_lidar_input = X[0]
    s008_coord_input = X[1]
    s008_y_train = y[0]

    (
        x_lidar_train,
        x_lidar_val,
        x_coord_train,
        x_coord_val,
        y_train,
        y_val,
    ) = train_test_split(
        s008_lidar_input,
        s008_coord_input,
        s008_y_train,
        test_size=0.2,
        random_state=trial.number,
        shuffle=True,
    )

    # ———————————————————————————————————————————————————————————————————————————— #

    model = None
    try:

        # ———————————————————————————————————————————————————————————————————————————— #
        #                              Model Construction                              #
        # ———————————————————————————————————————————————————————————————————————————— #

        # —————————————————————————————————— Scaler —————————————————————————————————— #
        # scaler = get_scaler(trial)
        # x_coord_train = scaler.fit_transform(x_coord_train)
        # x_coord_val = scaler.transform(x_coord_val)

        # ——————————————————————————————— LiDAR Input ——————————————————————————————— #
        # Input for LiDAR data (e.g., shape: (20, 200, 10))
        x_lidar_input = layers.Input(shape=(20, 200, 10))

        # ———————————————————————————————— GPS Input ———————————————————————————————— #
        # Input for coordinate data (e.g., shape: (2,))
        x_coord_input = layers.Input(shape=(x_coord_train.shape[1],))

        # Use z-score normalization
        norm_layer = layers.Normalization(axis=1, name="coord_input_normalization")

        # Computes the mean and variance of the input data
        norm_layer.adapt(x_coord_train)

        # Apply normalization to the input data
        x_coord_norm = norm_layer(x_coord_input)

        # Add spatial dimension
        x_coord_norm = layers.Reshape((1, 1, x_coord_input.shape[1]))(x_coord_norm)

        # Tile across the lidar grid
        # So the coordinates are repeated across the 20x200 grid
        x_coord_norm = layers.Lambda(lambda x: tf.tile(x, [1, 20, 200, 1]))(x_coord_norm)

        # ? If using scaler then uncomment the following line
        # x_coord_input = layers.Reshape((1, 1, x_coord_input.shape[1]))(x_coord_input)
        # ————————————————————————————— Combine Branches ————————————————————————————— #
        # Fuse channels: (batch,20,200,10) + (batch,20,200,2) → (batch,20,200,12)
        combined = layers.Concatenate(axis=-1)([x_lidar_input, x_coord_norm])
        

        # —————————————————————————————————— Output —————————————————————————————————— #
        outputs = layers.Dense(256, activation="softmax")(x)

        # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
        model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))
        
        # ———————————————————————————— Vizualize the Model ——————————————————————————— #      
        if show_summary:
            model.summary()

        if plot_model:
            # Display the model architecture image
            tf.keras.utils.plot_model(
                model,
                to_file=os.path.join(fig_dir, f"model_plot_{trial.number}.png"),
                show_shapes=True,
                show_layer_names=True,
            )
            display(Image(filename=os.path.join(fig_dir, f"model_plot_{trial.number}.png")))

        # ————————————————————————————— Compile the Model ———————————————————————————— #
        optimizer = get_optimizer(trial)
        model.compile(
            optimizer=optimizer,
            loss=losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"],
        )

        # ———————————————————————————————— Train Model ——————————————————————————————— #
        batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
        history = model.fit(
            [x_lidar_train, x_coord_train],
            y_train,
            validation_data=([x_lidar_val, x_coord_val], y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=get_callbacks(trial, checkpoint_dir),
            verbose=2,
        )

        model.save(os.path.join(model_dir, f"trial_{trial.number}.keras"))

        # ———————————————————————————————————————————————————————————————————————————— #
        #                            Penalize the Model Size                           #
        # ———————————————————————————————————————————————————————————————————————————— #
        loss = min(history.history["val_loss"])
        if size_penalizer == "flops":
            loss = troo.compute_flops_penalized_loss(loss=loss, model=model)
        elif size_penalizer == "params":
            loss = troo.compute_params_penalized_loss(loss=loss, model=model)

        # ———————————————————————————————————————————————————————————————————————————— #
        #                                 Trial Results                                #
        # ———————————————————————————————————————————————————————————————————————————— #
        clear_output(wait=True)

        epochs = list(range(1, len(history.history["loss"]) + 1))
        train_loss = history.history["loss"]
        val_loss = history.history["val_loss"]
        train_acc = history.history.get("accuracy", [])
        val_acc = history.history.get("val_accuracy", [])

        # Create figure with two subplots
        fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(16, 6))

        # Left: Loss
        ax_loss.plot(epochs, train_loss, marker="o", linestyle="-", label="Training Loss")
        ax_loss.plot(epochs, val_loss, marker="x", linestyle="--", label="Validation Loss")
        ax_loss.set_title("Training & Validation Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Loss")
        ax_loss.set_xticks(epochs)
        ax_loss.set_ylim(0, max(max(train_loss), max(val_loss)) * 1.05)
        ax_loss.grid(True)
        ax_loss.legend()

        # Right: Accuracy (if available)
        if train_acc and val_acc:
            ax_acc.plot(epochs, train_acc, marker="v", linestyle="-", label="Training Accuracy")
            ax_acc.plot(epochs, val_acc, marker="^", linestyle="--", label="Validation Accuracy")
            ax_acc.set_title("Training & Validation Accuracy")
            ax_acc.set_xlabel("Epoch")
            ax_acc.set_ylabel("Accuracy")
            ax_acc.set_xticks(epochs)
            ax_acc.set_ylim(0, 1)
            ax_acc.grid(True)
            ax_acc.legend()

            trial.set_user_attr("best_train_accuracy", float(max(train_acc)))
            trial.set_user_attr("best_val_accuracy", float(max(val_acc)))
        else:
            ax_acc.axis("off")  # hide if accuracy not present
            
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, f"trial_{trial.number}.png"), dpi=300)
        plt.close(fig)
            
        # ————————————————————————————— Evaluate on s009 ————————————————————————————— #
        test_loss, test_acc = model.evaluate(
            [s009_lidar_input, s009_coord_input],
            s009_y,
            batch_size=batch_size,
            verbose=0
        )
        
        trial.set_user_attr("test_accuracy_s009", float(test_acc))
        
        # ————————————————————————————— Print the results ———————————————————————————— #
        
        
        print(f"\n\n# ——————————————————————— Trial {trial.number} Results ——————————————————————— #")
        print("\n" + "="*15)
        print(f"Training loss: {loss:.12f}")
        print(f"Training accuracy: {max(train_acc):.4f}\n")
        print(f"Validation loss: {loss:.12f}")
        print(f"Validation accuracy: {max(val_acc):.4f}\n")
        print(f"Test loss (s009):     {test_loss:.12f}")
        print(f"Test accuracy (s009): {test_acc:.4f}")
        print("="*15 + "\n")
        print("# ———————————————————————————————————————————————————————————————————————————— #\n\n")

        return loss

    except optuna.exceptions.TrialPruned:
        raise  # simply propagate pruning
    except tf.errors.ResourceExhaustedError as oom_err:
        # Catch OOM / resource exhausted
        print(f"❌ Trial {trial.number} hit OOM (ResourceExhaustedError): {oom_err}")
        
        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(oom_err) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)
        
        return float("inf") # Return bad loss
    except Exception as e:
        print(f"An error occurred during the trial execution: {e}")
        traceback.print_exc()
        
        # Log the error to a file in the logs directory
        error_log_path = os.path.join(logs_dir, f"trial_{trial.number}_error.log")
        with open(error_log_path, "w") as log_file:
            log_file.write(f"Trial {trial.number} encountered an error:\n")
            log_file.write(str(e) + "\n\n")
            log_file.write("Traceback:\n")
            traceback.print_exc(file=log_file)

        return float("inf") # Return bad loss
    finally:
        if model is not None:
            clear_session()
            del model

## 7. Code Health Check

In [ ]:
resources_dir = os.path.join(RUN_DIR, "resources")
os.makedirs(resources_dir, exist_ok=True)
troo.log_resources(log_dir=resources_dir)

In [ ]:
_monitor_proc = troo.launch_kernel_monitor(custom_title="RUN_DIR", script_path="./tensoroo/_monitor_kernel_life.py")

## Main

In [ ]:
try:
    # ——————————————————————————————— Storage paths —————————————————————————————— #
    study_dir = os.path.join(RUN_DIR, "optuna_study")
    os.makedirs(study_dir, exist_ok=True)

    dirs = {
        "args": os.path.join(study_dir, "args"),
        "figures": os.path.join(study_dir, "figures"),
        "weights": os.path.join(study_dir, "weights"),
        "models": os.path.join(study_dir, "models"),
        "logs": os.path.join(study_dir, "logs"),
    }
    for path in dirs.values():
        os.makedirs(path, exist_ok=True)

    storage_path = f"sqlite:///{os.path.join(study_dir, 'optuna_study.db')}"
    checkpoint_dir, model_dir, fig_dir, args_dir, logs_dir = (
        dirs["weights"],
        dirs["models"],
        dirs["figures"],
        dirs["args"],
        dirs["logs"],
    )

    print(f"Initializing study at '{study_dir}'...")

    # —————————————————————————————————— Pruners ————————————————————————————————— #
    pruner = optuna.pruners.HyperbandPruner()

    # ——————————————————————————————————— Study —————————————————————————————————— #
    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=storage_path,
        direction="minimize",
        pruner=pruner,
        load_if_exists=True,
    )

    # Count trials done, then determine the remaining trials
    done_trials = len(
        study.get_trials(
            deepcopy=False,
            states=(
                optuna.trial.TrialState.COMPLETE,
                optuna.trial.TrialState.PRUNED,
                optuna.trial.TrialState.FAIL,
            ),
        )
    )
    n_remaining_trials = max(0, NUM_TRIALS - done_trials)

    study.optimize(
        lambda trial: objective(
            trial,
            X=[s008_lidar_input, s008_coord_input],
            y=[s008_y_train],
            checkpoint_dir=checkpoint_dir,
            model_dir=model_dir,
            fig_dir=fig_dir,
            logs_dir=logs_dir,
            epochs=EPOCHS,
            size_penalizer=None,
            use_regularization=False,
            residual_method=None,  #! Find your backbone first
        ),
        n_trials=n_remaining_trials,
        catch=(ValueError, RuntimeError),
        gc_after_trial=True,
        n_jobs=1,  # If you have multiple GPUs/Cores
        show_progress_bar=False,
    )

    # ————————————————————————————— Save Top-K Trials ———————————————————————————— #
    valid_trials = [
        t for t in study.trials
        if t.value is not None and not (math.isnan(t.value) or math.isinf(t.value))
    ]
    sorted_trials = sorted(valid_trials, key=lambda t: t.value)[:TOP_K]

    for rank, trial in enumerate(sorted_trials):
        trial_id = trial.number
        trial_params = trial.params
        trial_loss = trial.value
        trial_train_acc = trial.user_attrs.get("best_train_accuracy", None)
        trial_val_acc = trial.user_attrs.get("best_val_accuracy", None)
        trial_test_acc = trial.user_attrs.get("test_accuracy_s009", None)

        troo.save_trial_params_to_file(
            filepath=os.path.join(args_dir, f"top_{rank + 1}_trial.txt"),
            params=trial_params,
            rank=rank + 1,
            trial_id=trial_id,
            loss=trial_loss,
            val_accuracy=trial_val_acc,
            train_accuracy=trial_train_acc,
            test_accuracy=trial_test_acc,
            sampler=study.sampler.__class__.__name__,
        )

    # —————————————————————————— Clean-Up Non-Top Trials ————————————————————————— #
    all_trial_ids = {t.number for t in study.trials}
    top_trial_ids = {t.number for t in sorted_trials}

    cleanup_paths = [
        (checkpoint_dir, "trial_{trial_id}.weights.h5"),
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
    ]

    for trial_id in all_trial_ids - top_trial_ids:
        for base_dir, filename_template in cleanup_paths:
            file_path = os.path.join(base_dir, filename_template.format(trial_id=trial_id))
            if os.path.exists(file_path):
                os.remove(file_path)

    troo.analyze_study(study, fig_dir=fig_dir, table_dir=study_dir)

    # ————————————————————————————— End The Training ————————————————————————————— #
    failed_trials = sum(1 for t in study.trials if t.state != optuna.trial.TrialState.COMPLETE)
    troo.notify_training_success(
        recipients_file="./json/recipients.json",
        credentials_file="./json/credentials.json",
        subject=f"🎉 Training Complete - Failed Trials: {failed_trials}",
    )
except Exception as e:
    print(f"An error occurred: {e}")
    traceback.print_exc()

In [ ]:
# Kill the monitor kernel life process
if _monitor_proc is not None and _monitor_proc.poll() is None:
    os.killpg(_monitor_proc.pid, signal.SIGINT)